# Reproducing the Final Agents: Search Distillation for ZX-Calculus Circuit Optimisation

This notebook reproduces the **final agents** of the dissertation *end to end*: the
corrected Cuccaro adder training data, the search-distillation training pipeline
(blind chained search, behaviour-cloning distillation, guided search, fresh distillation),
and the Chapter 4 evaluation (best-of-10 protocol) for Agents **A** (adder-trained),
**R** (structured-random) and **P** (pure-random).

**Two ways to run it:**
1. **Quick verification (minutes):** load the released checkpoints and reproduce the
   evaluation numbers (Table 4.1 and the family means of Tables 4.3/4.4).
2. **Full retraining (about 30 min for Agent A and about 20 min each for R/P on an
   Apple M5, CPU):** set `RETRAIN = True` below.

**Requirements:** Python 3.12+, `pyzx==0.10.5`, `torch>=2.0`, `torch_geometric`, `pandas`.
Run from the repository root (`circopt-rl-zx-dissertation/`).

**A note on determinism:** training and the deterministic (argmax) rollouts are seeded
and reproduce exactly. The nine *sampled* rollouts of the best-of-10 protocol and the
two random circuit families are stochastic, so random-family and real-world numbers
reproduce in distribution rather than digit for digit (dissertation Section 3.11).

**Requirements.** Run this notebook from the root of the released repository; it uses
`src/` (environment, model, generators), `scripts/` (training, evaluation, certificates),
`tests/` (the adder regression test), `benchmarks/` (the Riu et al. circuit suite), and
`results/checkpoints/` (the three released agents). Install with
`pip install -r requirements.txt` on Python 3.10 or newer; `pyzx` is pinned at 0.10.5
because the environment reuses pyzx's verified rewrite internals. Adder and benchmark
results reproduce deterministically; the random-family draws use pyzx's unseedable
internal generator, so those circuits (and family means) differ between runs while the
qualitative pattern reproduces.


In [1]:
# 1. Setup: paths, seed, versions
import sys, random
from pathlib import Path

REPO_ROOT = Path.cwd()
assert (REPO_ROOT / "src" / "circopt_adder").exists(), "run this notebook from the repo root"
sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "scripts"))
sys.path.insert(0, str(REPO_ROOT))

import pyzx, torch
print("pyzx", pyzx.__version__, "| torch", torch.__version__)

from circopt_adder.config import DEVICE, SEED, Config, set_seed
set_seed(SEED)
print("device:", DEVICE, "| seed:", SEED)

RETRAIN = False   # set True to retrain all three agents from scratch

pyzx 0.10.5 | torch 2.12.0


device: cpu | seed: 42


## 2. Training data: the corrected Cuccaro ripple-carry adder

The generator implements the modulo-$2^n$ ripple-carry family of Cuccaro et al. (2004),
with the MAJ/UMA carry wiring corrected and verified (dissertation Section 3.3.3).
The cell below re-runs the exhaustive unitary correctness check for bit-widths 1-3:
every input pair must produce $b \leftarrow a + b \ (\mathrm{mod}\ 2^n)$.

In [2]:
# 2a. Exhaustive correctness check of the adder generator (seconds)
# Runs the repository's permanent regression test: for n = 1..3, every input pair
# must give B <- A + B (mod 2^n) with register A and the ancilla restored, and
# ancilla=1 must give A + B + 1 (Cuccaro et al. 2004, Sections 4.1-4.2).
import runpy, importlib.util
spec = importlib.util.spec_from_file_location(
    "test_adder_correctness", REPO_ROOT / "tests" / "test_adder_correctness.py")
tac = importlib.util.module_from_spec(spec); spec.loader.exec_module(tac)
for n in (1, 2, 3):
    tac.check(n)
    print(f"n={n}: OK ({4**n * 2} input/carry combinations verified)")

from circopt_adder.generators import ripple_carry_adder, _light_preprocess
from circopt_adder.zx_utils import two_qubit_gate_count

n=1: OK (8 input/carry combinations verified)
n=2: OK (32 input/carry combinations verified)
n=3: OK (128 input/carry combinations verified)


In [3]:
# 2b. Preprocessing: raw adders reduce to the 11n two-qubit / 10n T floor
for nb_ in (2, 3, 4, 10):
    raw = ripple_carry_adder(nb_)                      # pyzx Circuit
    prep = _light_preprocess(raw.to_graph())           # light preprocessing pipeline
    print(f"{nb_}-bit: raw 2Q = {two_qubit_gate_count(raw)}"
          f" -> preprocessed 2Q = {two_qubit_gate_count(prep)} (= 11 x {nb_})")

2-bit: raw 2Q = 32 -> preprocessed 2Q = 22 (= 11 x 2)
3-bit: raw 2Q = 48 -> preprocessed 2Q = 33 (= 11 x 3)
4-bit: raw 2Q = 64 -> preprocessed 2Q = 44 (= 11 x 4)
10-bit: raw 2Q = 160 -> preprocessed 2Q = 110 (= 11 x 10)


## 3. The search-distillation pipeline

The three steps of dissertation Section 3.8, implemented in
`scripts/train_agent_a_v3.py` (helpers) and `scripts/train_agent_a_v4_correct.py`
(the Agent A pipeline actually used):

1. **Blind chained search** - 150 uniform-random episodes per round, up to 3 rounds per
   bit-width; each round keeps the single best winning trajectory (cut at its best point,
   STOP appended) and restarts search from that circuit.
2. **Distillation** - behaviour-clone a fresh 128-channel GATv2 policy on the winning
   state-action pairs (cross-entropy, Adam, lr 1e-3).
3. **Guided search + fresh distillation** - repeat the chained search guided by the
   distilled policy (250 episodes per round, up to 4 rounds), then distil a *fresh*
   policy from the guided trajectories alone.

Agents R and P use the same recipe on their families (one winning trajectory per fresh
circuit; no guided round), via `scripts/train_agents_rp_v3.py`.

In [4]:
# 3a. (optional) Full retraining -- writes new checkpoints next to the released ones
if RETRAIN:
    import train_agent_a_v4_correct as v4   # ~30 min: blind + distill + guided + fresh distill
    v4.main()                               # saves results/checkpoints/agent_A_v4_correct.pt
    import train_agents_rp_v3 as rp3        # ~20 min per family
    rp3.main()                              # saves agent_v3_128_{structured,pure}_random.pt
else:
    print("RETRAIN = False: using the released checkpoints below.")

RETRAIN = False: using the released checkpoints below.


In [5]:
# 3b. Load the three final agents
from circopt_adder.model import ActorCriticGNN

def load_agent(name):
    cfg = Config(); cfg.gnn_channels = 128
    pol = ActorCriticGNN(cfg).to(DEVICE)
    pol.load_state_dict(torch.load(REPO_ROOT / "results" / "checkpoints" / name,
                                   map_location=DEVICE))
    pol.eval()
    return pol, cfg

agents = {
    "Agent A": load_agent("agent_A_v4_correct.pt"),
    "Agent R": load_agent("agent_v3_128_structured_random.pt"),
    "Agent P": load_agent("agent_v3_128_pure_random.pt"),
}
n_params = sum(p.numel() for n, p in agents["Agent A"][0].named_parameters()
               if not n.startswith("value"))   # policy path: input_proj + convs + policy_head
print(f"loaded {len(agents)} agents | policy-path parameters: {n_params:,}")

loaded 3 agents | policy-path parameters: 174,721


## 4. Evaluation: the Chapter 4 best-of-10 protocol

One deterministic (argmax) rollout plus nine sampled rollouts per (circuit, policy)
pair, keeping the lowest two-qubit count seen at any point in any episode. Agents
receive the light-preprocessed circuit, exactly as deployed (Section 3.7).

In [6]:
# 4a. Table 4.1 -- two-qubit count on adders, bit-widths 2-10 (~10 min on an M5)
import train_agent_a_v3 as v3
import pandas as pd

def best_of_10(policy, cfg, circ):
    best, _, _ = v3.episode(cfg, circ, policy=policy, greedy=True)
    for _ in range(9):
        m, _, _ = v3.episode(cfg, circ, policy=policy)
        best = min(best, m)
    return best

rows = []
for nb in range(2, 11):
    circ = _light_preprocess(ripple_carry_adder(nb).to_graph())
    row = {"bit-width": nb, "basic_opt (=start)": two_qubit_gate_count(circ)}
    for name, (pol, cfg) in agents.items():
        row[name] = best_of_10(pol, cfg, circ)
    rows.append(row)
    print(row)

table41 = pd.DataFrame(rows).set_index("bit-width")
table41
# Expected (Table 4.1): R and P equal the start on every width; A is exactly one
# below on every width, including the held-out 4- and 10-bit adders.

{'bit-width': 2, 'basic_opt (=start)': 22, 'Agent A': 21, 'Agent R': 22, 'Agent P': 22}


{'bit-width': 3, 'basic_opt (=start)': 33, 'Agent A': 32, 'Agent R': 33, 'Agent P': 33}


{'bit-width': 4, 'basic_opt (=start)': 44, 'Agent A': 43, 'Agent R': 44, 'Agent P': 44}


{'bit-width': 5, 'basic_opt (=start)': 55, 'Agent A': 54, 'Agent R': 55, 'Agent P': 55}


{'bit-width': 6, 'basic_opt (=start)': 66, 'Agent A': 65, 'Agent R': 66, 'Agent P': 66}


{'bit-width': 7, 'basic_opt (=start)': 77, 'Agent A': 76, 'Agent R': 77, 'Agent P': 77}


{'bit-width': 8, 'basic_opt (=start)': 88, 'Agent A': 87, 'Agent R': 88, 'Agent P': 88}


{'bit-width': 9, 'basic_opt (=start)': 99, 'Agent A': 98, 'Agent R': 99, 'Agent P': 99}


{'bit-width': 10, 'basic_opt (=start)': 110, 'Agent A': 109, 'Agent R': 110, 'Agent P': 110}


,basic_opt (=start),Agent A,Agent R,Agent P
bit-width,,,,
2,22,21,22,22
3,33,32,33,33
4,44,43,44,44
5,55,54,55,55
6,66,65,66,66
7,77,76,77,77
8,88,87,88,88
9,99,98,99,99
10,110,109,110,110


In [7]:
# 4b. Real-world arithmetic benchmarks (11 circuits; ~15 min on an M5)
from benchmarks.fetch_paper_circuits import load_all as load_paper

ARITH = ("add", "qcla", "mod")
rows = []
for name, c in sorted(load_paper().items()):
    if not any(k in name.lower() for k in ARITH):
        continue
    prep = _light_preprocess(c.to_basic_gates().to_graph())
    row = {"circuit": name, "start": two_qubit_gate_count(prep)}
    for aname, (pol, cfg) in agents.items():
        row[aname] = best_of_10(pol, cfg, prep)
    rows.append(row)
    print(row)

pd.DataFrame(rows).set_index("circuit")
# Compare with dissertation Table 4.2. Ties reproduce exactly; one-to-two-gate wins
# are found by sampled search and can vary by one gate between runs.

{'circuit': 'paper_Adder8', 'start': 243, 'Agent A': 243, 'Agent R': 243, 'Agent P': 243}


{'circuit': 'paper_QFTAdd8', 'start': 184, 'Agent A': 182, 'Agent R': 183, 'Agent P': 183}


{'circuit': 'paper_adder_8', 'start': 385, 'Agent A': 381, 'Agent R': 383, 'Agent P': 383}


{'circuit': 'paper_mod5_4', 'start': 27, 'Agent A': 27, 'Agent R': 27, 'Agent P': 26}


{'circuit': 'paper_mod_mult_55', 'start': 48, 'Agent A': 47, 'Agent R': 48, 'Agent P': 48}


...

{'circuit': 'paper_qcla_com_7', 'start': 174, 'Agent A': 174, 'Agent R': 174, 'Agent P': 172}


{'circuit': 'paper_qcla_mod_7', 'start': 366, 'Agent A': 366, 'Agent R': 366, 'Agent P': 366}


{'circuit': 'paper_rc_adder_6', 'start': 81, 'Agent A': 81, 'Agent R': 81, 'Agent P': 81}


{'circuit': 'paper_vbe_adder_3', 'start': 58, 'Agent A': 58, 'Agent R': 58, 'Agent P': 57}


,start,Agent A,Agent R,Agent P
circuit,,,,
paper_Adder8,243,243,243,243
paper_QFTAdd8,184,182,183,183
paper_adder_8,385,381,383,383
paper_mod5_4,27,27,27,26
paper_mod_mult_55,48,47,48,48
paper_mod_red_21,105,105,105,104
paper_qcla_adder_10,209,209,209,209
paper_qcla_com_7,174,174,174,172
paper_qcla_mod_7,366,366,366,366


### 4c. Random families and the untrained-network control

Tables 4.3 to 4.5 report family means on 10 structured-random and 10 pure-random
circuits, and Section 5.3.1 rests on one comparison: on these families the final
Agents R and P perform no better than a network with random, untrained weights
evaluated under the same best-of-10 protocol. This cell draws fresh circuits from the
two generators and runs that comparison (roughly 20 minutes on an M5).

In [ ]:
# 4c. Random families (Tables 4.3 to 4.5) and the untrained control (Section 5.3.1)
# Fresh draws: pyzx's internal RNG is unseedable, so family MEANS move by a few points
# between runs while the ordering pattern reproduces.
import evaluate_random10 as er
from circopt_adder.generators import (make_random_circuit_generator,
                                      make_pure_random_circuit_generator)

ucfg = Config(); ucfg.gnn_channels = 128
untrained = ActorCriticGNN(ucfg).to(DEVICE); untrained.eval()   # the ch5 control

gen_s = make_random_circuit_generator(ucfg.n_qubits, ucfg.n_gates_random, seed=SEED + 1)
gen_p = make_pure_random_circuit_generator(ucfg.pure_random_min_qubits,
    ucfg.pure_random_max_qubits, ucfg.pure_random_min_gates,
    ucfg.pure_random_max_gates, seed=SEED + 1)
families = {"structured-random": [gen_s() for _ in range(10)],
            "pure-random":       [gen_p() for _ in range(10)]}

policies = dict(agents); policies["Untrained control"] = (untrained, ucfg)
summary = []
for fam, circs in families.items():
    red = {name: [] for name in ["basic_opt"] + list(policies)}
    for circuit in circs:
        raw2q = two_qubit_gate_count(circuit)
        out = er.baseline_basic_optimization(circuit)
        red["basic_opt"].append(100 * (raw2q - two_qubit_gate_count(out)) / raw2q)
        prep = _light_preprocess(circuit.to_graph() if hasattr(circuit, "to_graph") else circuit)
        for name, (pol, pcfg) in policies.items():
            best = er.best_of_n(pol, prep, pcfg)
            red[name].append(100 * (raw2q - two_qubit_gate_count(best)) / raw2q)
    summary.append({"family": fam, **{k: round(sum(v) / len(v), 1) for k, v in red.items()}})

pd.DataFrame(summary).set_index("family")
# Expected (Table 4.3 pattern): every agent clears basic_opt by several points through
# best-of-10 sampling, and Agents R and P land within a few points of the untrained
# control -- the Section 5.3.1 finding that on these families the margin comes from
# sampled search, not from anything learned.

## 5. What Agent A learned: the pivot-gadget motif

A single greedy rollout on any adder shows the learned behaviour directly: three
pivot-gadget (PIVG) applications whose gain lands only when the sequence completes
(dissertation Section 5.2.3). Exhaustive enumeration of all PIVG sequences up to
length 3 certifies this result optimal within the motif class on the 2-, 3- and 4-bit
adders (`scripts/motif_experiments.py`).

In [8]:
# 5a. Trace the greedy episode on the held-out 4-bit adder
from circopt_adder.env import ZXOptEnv

pol, cfg = agents["Agent A"]
circ = _light_preprocess(ripple_carry_adder(4).to_graph())
env = ZXOptEnv(lambda: circ, cfg)
obs, _ = env.reset()
trace, prev_best, t_best = [], env.best_metric, -1
for t in range(cfg.max_episode_steps):
    with torch.no_grad():
        logits, _ = pol.forward(obs.to(DEVICE))
    a = int(torch.argmax(logits).item())
    act = env._action_index[a]
    trace.append(act if act == "STOP" else act[0])
    obs, r, term, trunc, info = env.step(a)
    if env.best_metric < prev_best:
        prev_best, t_best = env.best_metric, t
    if term or trunc:
        break
print("start 2Q:", two_qubit_gate_count(circ), "| best 2Q:", env.best_metric)
print("actions up to the best point:", trace[: t_best + 1])
# Expected: 44 -> 43 on a width never seen in training, produced by three
# pivot-gadget (PIVG) applications whose gain lands only when the sequence
# completes -- the certified motif of dissertation Section 5.2.3.

start 2Q: 44 | best 2Q: 43
actions up to the best point: ['PIVG', 'PIVG', 'PIVG']


In [ ]:
# 5b. (optional) Exhaustive certificate for the rewrite sequence class (Section 5.2.2)
# Enumerates EVERY sequence of up to three pivot-gadget applications on the 2-bit adder
# and confirms nothing in the class beats the agent's 21 (56,715 sequences; minutes to
# tens of minutes). The dissertation's 3-bit (198,892) and held-out 4-bit (480,949)
# certificates run the same function at nb=3 and nb=4 (see scripts/motif_experiments.py,
# experiment1; those take hours).
CERTIFY = False
if CERTIFY:
    import motif_experiments as me
    circ2 = _light_preprocess(ripple_carry_adder(2).to_graph())
    start = two_qubit_gate_count(circ2)
    g2 = me.circuit_to_graphlike(circ2)
    best, n_seq = me.exhaustive_triples(g2, start)
    print(f"2-bit adder: start={start}, best over {n_seq:,} sequences = {best}")
    assert best == 21, "certificate should reproduce the agent's 21"
else:
    print("CERTIFY = False: enumeration skipped; see Section 5.2.2 and scripts/motif_experiments.py.")

## 6. Provenance

| Artefact | Path |
|---|---|
| Agent A checkpoint | `results/checkpoints/agent_A_v4_correct.pt` |
| Agent R checkpoint | `results/checkpoints/agent_v3_128_structured_random.pt` |
| Agent P checkpoint | `results/checkpoints/agent_v3_128_pure_random.pt` |
| Chapter 4 evaluation data | `results/logs/evaluation_results_final_v4.csv` |
| Agent A training script | `scripts/train_agent_a_v4_correct.py` |
| R/P training script | `scripts/train_agents_rp_v3.py` |
| Adder regression test | `tests/test_adder_correctness.py` |
| Motif certificates | `scripts/motif_experiments.py` |
| Benchmark suite (Riu et al. 2025) | `benchmarks/Original/` |
| Random-family evaluation script | `scripts/evaluate_random10.py` |
| Pinned software versions | `requirements.txt` |

Training hardware for all reported runs: Apple MacBook Air, M5 (10 CPU cores), 24 GB
unified memory; no GPU used.